In [ ]:
import os; os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [2]:
import torch


In [ ]:
import math
import torch.nn as nn


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


In [5]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "tiny_shakespeare.txt"
)
print("Downloaded!")


Downloaded!


In [6]:
text = ""
# with open("file.txt", 'r', encoding='utf-8') as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         content = line.split(sep = '-', maxsplit=1)
#         if '<Media omitted>' in content:
#             continue
#         if len(content)<2:
#             text += "\n" + line
#             continue
#         text += "\n"
#         text += content[1].strip()

with open("tiny_shakespeare.txt", 'r', encoding='utf-8') as f:
    for line in f:
        text += line

chars = list(set(text))
chars.sort()
stoi = {char : ind for ind, char in enumerate(chars)}
itos = {ind : char for ind, char in enumerate(chars)}
vocab_size = len(stoi)

def encode(text):
    return [stoi[char] for char in text]

def decode(embedding):
    return [itos[ind] for ind in embedding]

def train_test_split(data, train=0.9):
    n = int(train*len(data))
    tr = data[:n]
    val = data[n:len(data)]
    return tr, val


In [ ]:
from bpe import BPETokenizer


In [9]:
bpetokenizer = BPETokenizer(vocab_size=1000)
bpetokenizer.train(text)


Number of byte-pair scans over entire data = 1


In [ ]:
print(text[:1000])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [11]:
# data = torch.tensor(encode(text), dtype=torch.long)
data = torch.tensor(bpetokenizer.encode(text), dtype=torch.long)
train_data, val_data = train_test_split(data)

torch.manual_seed(42)
batch_size = 64
block_size = 256

def get_batch(split):
    data = train_data if split=='train' else val_data
    inds = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in inds]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in inds]).to(device)
    return x, y

xb, yb = get_batch('train')
print(xb)
print(yb)


tensor([[100, 298, 579,  ..., 289, 703, 339],
        [321, 898, 297,  ..., 322, 290, 342],
        [ 10, 851, 292,  ..., 121,  98, 364],
        ...,
        [364, 308, 400,  ..., 292, 945,  44],
        [373, 311,  58,  ..., 467, 328, 403],
        [488, 288, 533,  ..., 325, 608, 103]], device='cuda:0')
tensor([[298, 579, 404,  ..., 703, 339,  46],
        [898, 297, 367,  ..., 290, 342, 122],
        [851, 292,  44,  ...,  98, 364, 116],
        ...,
        [308, 400, 366,  ..., 945,  44, 288],
        [311,  58, 277,  ..., 328, 403,  44],
        [288, 533, 312,  ..., 608, 103, 371]], device='cuda:0')


In [ ]:
import math


In [ ]:
def cosinedecaylr(step, total_steps, warmup_steps=100, initial_lr=0.0, max_lr=3e-4, min_lr=0.0):
    if step <= warmup_steps:
        return (max_lr-initial_lr)*step/warmup_steps
    else:
        return math.cos(math.pi/2*(step-warmup_steps)/(total_steps-warmup_steps))*(max_lr-min_lr) + min_lr


In [14]:
class AdamOptimizer:
    def __init__(self, params, lr=3e-4, eps=1e-8, betas=(0.9, 0.999)):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.beta1 = betas[0]
        self.beta2 = betas[1]
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self, regularization=None, weight_decay=0.1):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                if regularization=='L2':
                    p.grad = p.grad + weight_decay*p
                self.m[i] = self.beta1*self.m[i] + (1-self.beta1)*p.grad
                m_hat = self.m[i]/(1-self.beta1**self.t)
                self.v[i] = self.beta2*self.v[i] + (1-self.beta2)*(p.grad**2)
                v_hat = self.v[i]/(1-self.beta2**self.t)
                if regularization=='weight_decay':
                    p -= self.lr*weight_decay*p
                p -= (m_hat*self.lr)/(v_hat.sqrt()+self.eps)

    
    def set_lr(self, lrate):
        self.lr = lrate
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [ ]:
def softmax(x):
    exp_x = torch.exp(x)
    return exp_x/exp_x.sum(dim=-1, keepdim=True) # why -1? -> the matrix is batchxqueryxkey, so we have to softmax over all keys

def causal_softmax(x): # matrix = (batch, block_size, block_size) - attention matrix
    mask = torch.tril(torch.ones(block_size, block_size)).to(device)
    x_mask = x.masked_fill(mask == 0, -float('inf'))
    return softmax(x_mask)  


In [ ]:
n_embd = 384

class Attention(nn.Module):
    def __init__(self, heads=6):
        super().__init__()
        self.heads = heads
        self.head_size = n_embd//self.heads
        self.W_q = nn.Linear(n_embd, n_embd, bias=False)
        self.W_k = nn.Linear(n_embd, n_embd, bias=False)
        self.W_v = nn.Linear(n_embd, n_embd, bias=False)
        self.W_o = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))) # register_buffer - tells pytorch that it is not trainable
    
    def multi_head(self, mat):
        B = mat.shape[0] # batch_size
        mat = mat.view(B, block_size, self.heads, self.head_size)
        return mat.transpose(1, 2)  # (B, heads, T, head_size)
    
    def forward(self, batch): # batch = (batch_size, block_size, embedding_size), W_q = (embedding_size, output_size=n_embd)
        B = batch.shape[0]
        Q = self.W_q(batch) # (batch_size, block_size, n_embd)
        K = self.W_k(batch)
        V = self.W_v(batch)
        Q = self.multi_head(Q)
        K = self.multi_head(K)
        V = self.multi_head(V) 
        w = Q@K.transpose(-2, -1) # we only transpose within a head within a block - (B, heads, T, T)
        w = w/math.sqrt(self.head_size) 
        # W = causal_softmax(w2) # (batch_size, heads, block_size, block_size)
        w = w.masked_fill(self.mask == 0, -float('inf'))
        W = torch.softmax(w, dim=3) # w - (batch_size, heads, block_size, block_size) - we softmax across fourth dim
        new_tokens = W @ V # (B, heads, T, head_size)
        new_tokens = new_tokens.transpose(1, 2) # (B, T, heads, head_size)
        new_tokens = new_tokens.reshape(B, block_size, -1)
        new_tokens = self.W_o(new_tokens)
        return new_tokens


In [ ]:
def relu(x): return torch.clamp(x, min=0)
def gelu(x): return x*0.5*(1+torch.erf(x/math.sqrt(2)))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 =  nn.Linear(n_embd, 4*n_embd, bias=True)
        self.l2 = nn.Linear(4*n_embd, n_embd, bias=True)

    def forward(self, batch):
        x1 = self.l1(batch)
        x2 = gelu(x1)
        x3 = self.l2(x2)
        return x3


In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = Attention()
        self.attention_norm = nn.LayerNorm(n_embd)
        self.ff = FeedForward()
        self.ff_norm = nn.LayerNorm(n_embd)
        self.dropout = nn.Dropout(0.2) # zeroes out some of the activations (tensor values)
    
    def forward(self, batch):
        batch = batch + self.dropout(self.attention(self.attention_norm(batch))) # nn.Module runs self.forward automatically
        batch = batch + self.dropout(self.ff(self.ff_norm(batch)))
        return batch
    

In [ ]:
def loss(logits, y):
    loss_fn = nn.CrossEntropyLoss()
    return loss_fn(logits.view(-1, bpetokenizer.vocab_size), y.view(-1))


In [20]:
class Model(nn.Module):
    def __init__(self, layers=4):
        super().__init__()
        self.n_layers = layers
        self.token_embedding = nn.Embedding(bpetokenizer.vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block() for _ in range(self.n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, bpetokenizer.vocab_size, bias=True)
    
    def forward(self, batch): # batch = [[8 numbers], [], [], []] - 4 batches
        tokens = self.token_embedding(batch) # (batch_size, block_size, n_embd)
        tokens = tokens + self.position_embedding(torch.arange(block_size).to(device)) # (batch, block, n_embd) + (block, n_embd)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.ln_f(tokens)
        logits = self.lm_head(tokens)
        return logits
    



In [21]:
def train_step(model : Model, optimizer, step, total_steps):
    model.train()
    optimizer.zero_grad()
    xb, yb = get_batch('train')
    logits = model(xb)
    step_loss = loss(logits, yb)
    step_loss.backward()
    lr = cosinedecaylr(step=step, total_steps=total_steps)
    if isinstance(optimizer, AdamOptimizer):
        optimizer.set_lr(lr)
    else:
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
    optimizer.step()
    return step_loss.item()

def val_step(model):
    model.eval()
    with torch.no_grad():
        xb, yb = get_batch('val')
        logits = model(xb)
        step_loss = loss(logits, yb)
    return step_loss.item()

def accuracy(model, samples=1000):
    ...

def epoch(model, optimizer):
    val_interval = 100
    steps = len(train_data)//(batch_size*block_size)
    net_tr_loss = 0
    net_val_loss = 0
    for i in range(1, steps+1):
        tr_loss = train_step(model, optimizer)
        net_tr_loss += tr_loss
        if i%val_interval==0:
            val_loss = val_step(model)
            net_val_loss += val_loss
    return net_tr_loss/steps, net_val_loss/(steps//val_interval)

def train(epochs=1):
    model = torch.compile(Model().to(device))
    # model = Model().to(device)
    # optimizer = AdamOptimizer(model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    for n_epoch in range(1, epochs+1):
        tr_loss, val_loss = epoch(model, optimizer)
        print(f"Epoch {n_epoch} | Train Loss: {tr_loss} | Val Loss: {val_loss}")
    return model

def find_optimizer(model, lr=3e-4, regularization=None, weight_decay=0.1):
    if not regularization:
        return torch.optim.Adam(model.parameters(), lr=lr)
    elif regularization=="L2":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif regularization=="weight_decay":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def train_iters(max_iters=3400, val_interval=200, regularization=None, weight_decay=0.1):
    model = torch.compile(Model().to(device))
    # model = Model().to(device)
    # optimizer = AdamOptimizer(model.parameters())
    optimizer = find_optimizer(model, regularization=regularization, weight_decay=weight_decay)
    best_val_loss = float('inf')
    for n_iter in range(1, max_iters+1):
        tr_loss = train_step(model, optimizer, n_iter, max_iters)
        if n_iter%val_interval==0:
            val_loss = val_step(model)
            print(f"Step {n_iter:5d}/{max_iters} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f}")
            if val_loss < best_val_loss:
                torch.save(model.state_dict(), "best_model.pt")
                best_val_loss = val_loss
    model.load_state_dict(torch.load("best_model.pt"))
    return model, best_val_loss

torch.manual_seed(42)
model1, val_loss = train_iters(max_iters=3400, regularization="L2")
torch.manual_seed(42)
model2, val_loss = train_iters(max_iters=3400, regularization="weight_decay")


W0715 20:30:04.737000 2175 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Step   200/3400 | Train Loss: 5.8320 | Val Loss: 5.7851
Step   400/3400 | Train Loss: 5.7849 | Val Loss: 5.7103
Step   600/3400 | Train Loss: 5.7621 | Val Loss: 5.7358
Step   800/3400 | Train Loss: 5.8122 | Val Loss: 5.7852
Step  1000/3400 | Train Loss: 5.8419 | Val Loss: 5.8294
Step  1200/3400 | Train Loss: 5.8745 | Val Loss: 5.8447
Step  1400/3400 | Train Loss: 5.9205 | Val Loss: 5.8732
Step  1600/3400 | Train Loss: 5.9557 | Val Loss: 5.9098
Step  1800/3400 | Train Loss: 6.0008 | Val Loss: 5.9351
Step  2000/3400 | Train Loss: 5.9880 | Val Loss: 5.9688
Step  2200/3400 | Train Loss: 6.0518 | Val Loss: 6.0169
Step  2400/3400 | Train Loss: 6.0781 | Val Loss: 6.0105
Step  2600/3400 | Train Loss: 6.1053 | Val Loss: 6.0749
Step  2800/3400 | Train Loss: 6.1207 | Val Loss: 6.0777
Step  3000/3400 | Train Loss: 6.1325 | Val Loss: 6.1028
Step  3200/3400 | Train Loss: 6.1588 | Val Loss: 6.1200
Step  3400/3400 | Train Loss: 6.1502 | Val Loss: 6.1315
Step   200/3400 | Train Loss: 4.1305 | Val Loss:

In [27]:
print(val_loss)


3.5042152404785156


In [28]:
def generate(model, max_new_tokens=1000, temperature=1):
    model.eval()
    idx = torch.zeros((1, 1), dtype=torch.long).to(device)
    for _ in range(max_new_tokens):
        if idx.shape[1] < block_size:
            pad = torch.zeros(1, block_size - idx.shape[1], dtype=torch.long).to(device)
            idx_cond = torch.cat([pad, idx], dim=1)
        else:
            idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature  # last token, apply temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return ''.join(bpetokenizer.decode(idx[0].tolist()))

print(generate(model))


 st
How outide it:
So fainly actly you play then.

LUCIO:
Come, come, I l have, sir.

LARIA:
You think, sir, and you of this is a man and that
scombit nothing; pompering your presettful
po.

IARCIUS:
Procession lies. in that, Jion you.

JULIET:
The case you tunes of yours indect. What is this?

DUKE VINCENTIO:
That's the necesst was so, whose impine
staulate cover fly express of this: against him
By any in his memioning dispersem; and here Lutes
Oponoys where and news made worthy
Are all burning it is not.

DUKE VINCENTIO:
Though misster, yet you say, mude;
I know not.

Provost:
I tell you well;
You had I would have bity going well mission
With a shameful former drown: though I must flyoise.

JULIET:
Ay, by your life of pardons: the whips.

JULIET:
What's the queen?

Nurse:
Thou didst swear thee merry against thy way,
And thy face as to bed.
Nay, you dispatch him in my sister pride
Matches again; and now thereween past those those
ges me from mine enemies
Marry that my life to kind sle